In [ ]:
"""Day 19 / Session 3 - Lab Assignment 2: Scoring a Detector - Precision/Recall/F1 at an IoU Threshold."""


def iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    inter_w = max(0, min(ax2, bx2) - max(ax1, bx1))
    inter_h = max(0, min(ay2, by2) - max(ay1, by1))
    inter = inter_w * inter_h
    union = (ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - inter
    if union == 0:
        return 0.0
    return float(inter / union)


def match_predictions(pred_boxes, pred_scores, gt_boxes, iou_thresh=0.5):
    """Greedy matching: highest-scoring prediction first, each ground-truth box can be matched only once."""
    order = sorted(range(len(pred_boxes)), key=lambda i: -pred_scores[i])
    used = set()
    tp = 0
    fp = 0
    for i in order:
        best_iou = 0.0
        best_gt = None
        for j in range(len(gt_boxes)):
            if j in used:
                continue
            value = iou(pred_boxes[i], gt_boxes[j])
            if value > best_iou:
                best_iou = value
                best_gt = j
        if best_gt is not None and best_iou >= iou_thresh:
            tp += 1
            used.add(best_gt)
        else:
            fp += 1
    fn = len(gt_boxes) - len(used)               # ground truth nobody matched
    return tp, fp, fn


def precision_recall(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return precision, recall


def f1_score(precision, recall):
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)
